In [1]:
import sunpy
import numpy as np
from math import *
import astropy.units as u
from astropy.io import fits
import matplotlib.pyplot as plt
from sunpy.coordinates import frames
import matplotlib.gridspec as gridspec
from scipy.interpolate import interp1d,RegularGridInterpolator
from astropy.coordinates import SkyCoord
from scipy.io import readsav
from matplotlib.patches import Rectangle
from scipy.ndimage import zoom
from astropy.wcs import WCS
from astropy.wcs.utils import pixel_to_skycoord
import cv2
import glob
import os
from astropy.coordinates import SkyCoord
from sunpy.physics.differential_rotation import differential_rotate
from sunpy.coordinates import propagate_with_solar_surface
import pandas as pd

In [ ]:
from pathlib import Path
import gc
import numpy as np
import astropy.units as u
from astropy.io import fits
from astropy.wcs.utils import pixel_to_skycoord
import sunpy.map
from sunpy.coordinates import frames


# ==================== 1. 路径和参数 ====================
root = Path(r"C:\Learning\PHD3rd\sunspotscar\scar_for_20240618")
root = Path(r'E:\Learning\PHD3rd\sunspotscar\data\to_be_exam\20130411_065500_UTC')
input_dir = root / "hmi.B_720s"

output_dirs = {
    "Bxyz": root / "hmi_bxyz",
    "Br": root / "hmi_Br",
    "Btheta": root / "hmi_Btheta",
    "Bphi": root / "hmi_Bphi",
    "inclination": root / "inclination",
}

block_rows = 256       # 每次处理 256 行，减少全日面坐标转换的内存占用
limit = 10000.0        # 超过此范围的磁场分量标为 NaN，单位 G
overwrite = False     # 如需覆盖已经生成的文件，改为 True


# ==================== 2. 翻转后同步更新坐标头 ====================
def rotated_meta(base):
    """对应 np.rot90(data, 2)，只翻转像素，不进行插值。"""
    ny, nx = base.data.shape
    meta = base.meta.copy()
    meta["crpix1"] = nx + 1 - base.meta["crpix1"]
    meta["crpix2"] = ny + 1 - base.meta["crpix2"]

    for key in ["crota1", "crota2", "blank", "bscale", "bzero", "datamin", "datamax", "checksum", "datasum"]:
        meta.pop(key, None)

    meta["pc1_1"], meta["pc1_2"], meta["pc2_1"], meta["pc2_2"] = (-base.wcs.wcs.pc).ravel()
    meta["bunit"] = "G"
    meta["rot180"] = True
    return meta


# ==================== 3. 原始矢量磁图 → Bxyz ====================
def make_bxyz(paths, stamp):
    base = sunpy.map.Map(paths["field"])
    B = base.data.astype(np.float32)
    inc = fits.getdata(paths["inclination"], memmap=False).astype(np.float32)
    azi = fits.getdata(paths["azimuth"], memmap=False).astype(np.float32)
    dis = fits.getdata(paths["disambig"], memmap=False)

    if not (B.shape == inc.shape == azi.shape == dis.shape):
        raise ValueError(f"{stamp}: 四个原始数组的尺寸不一致")

    # 原始 inclination 是相对视线方向的角度，不是最后要计算的局部倾角。
    bad = (~np.isfinite(B)) | (~np.isfinite(inc)) | (~np.isfinite(azi)) | (~np.isfinite(dis))
    bad |= (B < 0) | (B > limit) | (inc < 0) | (inc > 180)

    # 沿用参考 notebook 的消歧方案和 Bxyz 定义。
    azi[dis > 3] += 180.0
    gamma = np.deg2rad(inc)
    psi = np.deg2rad(azi + 270.0)

    Bx = B * np.sin(gamma) * np.cos(psi)
    By = B * np.sin(gamma) * np.sin(psi)
    Bz = B * np.cos(gamma)

    # 无效像素统一设为 NaN，避免继续生成看似有效的倾角。
    Bx[bad], By[bad], Bz[bad] = np.nan, np.nan, np.nan
    Bx = np.rot90(Bx, 2).copy()
    By = np.rot90(By, 2).copy()
    Bz = np.rot90(Bz, 2).copy()

    meta = rotated_meta(base)
    output_path = output_dirs["Bxyz"] / f"hmi.B_720s.{stamp}.Bxyz.fits"

    # 一个 FITS 保存三个分量，可用 fits.getdata(path, "Bx") 等方式读取。
    hdus = []
    for index, (name, data) in enumerate([("Bx", Bx), ("By", By), ("Bz", Bz)]):
        component_meta = meta.copy()
        component_meta["bcomp"] = name
        component_meta["content"] = f"HMI {name}"
        header = sunpy.map.Map(data, component_meta).fits_header
        header["EXTNAME"] = name
        hdu = fits.PrimaryHDU(data, header) if index == 0 else fits.ImageHDU(data, header, name=name)
        hdus.append(hdu)

    with fits.HDUList(hdus) as hdul:
        hdul.writeto(output_path, overwrite=True)

    return output_path


# ==================== 4. Bxyz → Br、Btheta、Bphi → 倾角 ====================
def make_brthetaphi_inclination(bxyz_path, stamp):
    with fits.open(bxyz_path, memmap=True) as hdul:
        B_map = sunpy.map.Map(hdul["Bx"].data, hdul["Bx"].header)
        ny, nx = B_map.data.shape
        components = {name: np.full((ny, nx), np.nan, dtype=np.float32) for name in ["Br", "Btheta", "Bphi", "inclination"]}

        # 从翻转后的 WCS 获取坐标参数，不再额外给 p 加 π。
        phi0 = B_map.observer_coordinate.lon.to_value(u.rad)
        b = B_map.observer_coordinate.lat.to_value(u.rad)
        pc = B_map.wcs.wcs.pc
        p = np.arctan2(pc[0, 1], pc[0, 0])
        sb, cb, sp, cp = np.sin(b), np.cos(b), np.sin(p), np.cos(p)

        for y0 in range(0, ny, block_rows):
            y1 = min(y0 + block_rows, ny)
            rows = slice(y0, y1)
            Bx, By, Bz = [hdul[name].data[rows].astype(np.float64) for name in ["Bx", "By", "Bz"]]

            # 每个像素对应的太阳经纬度；日面外位置会产生 NaN。
            y, x = np.mgrid[y0:y1, 0:nx]
            coords = pixel_to_skycoord(x, y, B_map.wcs)
            hg = coords.transform_to(frames.HeliographicStonyhurst)
            dphi = hg.lon.to_value(u.rad) - phi0
            lam = hg.lat.to_value(u.rad)
            sl, cl, sd, cd = np.sin(lam), np.cos(lam), np.sin(dphi), np.cos(dphi)

            # 沿用参考 notebook 的变换矩阵。
            k11 = cl * (sb * sp * cd + cp * sd) - sl * cb * sp
            k12 = -cl * (sb * cp * cd - sp * sd) + sl * cb * cp
            k13 = cl * cb * cd + sl * sb
            k21 = sl * (sb * sp * cd + cp * sd) + cl * cb * sp
            k22 = -sl * (sb * cp * cd - sp * sd) - cl * cb * cp
            k23 = sl * cb * cd - cl * sb
            k31 = -sb * sp * sd + cp * cd
            k32 = sb * cp * sd + sp * cd
            k33 = -cb * sd

            # Br 向外为正，Btheta 向南为正，Bphi 向西为正。
            Br = (k11 * Bx + k12 * By + k13 * Bz).astype(np.float32)
            Btheta = (k21 * Bx + k22 * By + k23 * Bz).astype(np.float32)
            Bphi = (k31 * Bx + k32 * By + k33 * Bz).astype(np.float32)

            bad = (~np.isfinite(Bx)) | (~np.isfinite(By)) | (~np.isfinite(Bz))
            for arr in [Br, Btheta, Bphi]:
                bad |= (~np.isfinite(arr)) | (np.abs(arr) > limit)
            for arr in [Br, Btheta, Bphi]:
                arr[bad] = np.nan

            # 相对局部径向轴的倾角：竖直 0°，水平 90°，不区分磁极性。
            Bh = np.hypot(Btheta, Bphi)
            inclination = np.degrees(np.arctan2(Bh, np.abs(Br))).astype(np.float32)
            inclination[(Bh == 0) & (Br == 0)] = np.nan  # 零磁场没有确定方向

            components["Br"][rows] = Br
            components["Btheta"][rows] = Btheta
            components["Bphi"][rows] = Bphi
            components["inclination"][rows] = inclination

        # 各个物理量分别保存，保留与数组匹配的坐标头和原始观测时间。
        for name, data in components.items():
            meta = B_map.meta.copy()
            meta.pop("extname", None)
            meta["bunit"] = "deg" if name == "inclination" else "G"
            meta["bcomp"] = name
            meta["content"] = "Local inclination, 0-90 deg" if name == "inclination" else f"HMI {name}"
            output_path = output_dirs[name] / f"hmi.B_720s.{stamp}.{name}.fits"
            sunpy.map.Map(data, meta).save(output_path, overwrite=True)


# ==================== 5. 检查并处理全部 21 个时刻 ====================

# input_dir：原始数据文件夹。
# field_files：所有 field.fits 文件的路径列表；每个文件对应一个时刻。
# glob() 按文件名模式查找文件，sorted() 按文件名排序，这批文件也就按时间排好了。
field_files = sorted(input_dir.glob("hmi.B_720s.*.field.fits"))
# if len(field_files) != 21:
#     raise ValueError(f"预计有 21 组数据，实际找到 {len(field_files)} 个 field 文件，请检查输入目录")

# records：保存每个时刻的时间标记，以及该时刻所需的四个原始文件路径。
records = []
for field_path in field_files:
    # field_path：当前这个 field.fits 文件的完整路径；field_path.name 只取文件名。
    # 去掉文件名前后固定的部分，得到时间标记，例如 20240618_200000_TAI。
    stamp = field_path.name.removeprefix("hmi.B_720s.").removesuffix(".field.fits")

    # paths：字典，根据同一个时间标记拼出四个文件路径。
    # 例如 paths["field"] 是场强文件路径，paths["azimuth"] 是方位角文件路径。
    paths = {name: input_dir / f"hmi.B_720s.{stamp}.{name}.fits" for name in ["field", "inclination", "azimuth", "disambig"]}

    # 先检查四个文件是否齐全，缺少任何一个就停止并指出文件名。
    for path in paths.values():
        if not path.is_file():
            raise FileNotFoundError(f"缺少文件：{path}")

    # 将当前时刻的时间标记和文件路径放进 records，供后面的处理循环使用。
    records.append((stamp, paths))

# output_dirs：前面定义的输出文件夹字典；values() 取出其中的所有文件夹路径。
# parents=True：上级文件夹不存在时也一起创建。
# exist_ok=True：文件夹已经存在时不报错。
for folder in output_dirs.values():
    folder.mkdir(parents=True, exist_ok=True)

# 依次处理每个时刻：
# index 是处理序号，从 1 开始；stamp 是时间标记；paths 是该时刻的四个输入文件路径。
for index, (stamp, paths) in enumerate(records, start=1):
    print(f"[{index}] 正在处理 {stamp}", flush=True)  # flush=True 让进度及时显示

    # 第一步：读取原始矢量磁图，生成并保存 Bxyz。
    # bxyz_path：生成的 Bxyz.fits 文件的完整路径。
    bxyz_path = make_bxyz(paths, stamp)
    gc.collect()  # 尝试回收已经不再使用的对象，减少连续处理时的内存占用

    # 第二步：读取刚保存的 Bxyz，计算并保存 Br、Btheta、Bphi 和局部倾角。
    make_brthetaphi_inclination(bxyz_path, stamp)
    gc.collect()

    print(f"[{index}] 已保存 Bxyz、Br、Btheta、Bphi、inclination", flush=True)

print("全部处理完成！文件名中的时间保留原始 TAI 标记。")

[1/21] 正在处理 20130411_062400_TAI
[1] 已保存 Bxyz、Br、Btheta、Bphi、inclination
[2/21] 正在处理 20130411_063600_TAI
[2] 已保存 Bxyz、Br、Btheta、Bphi、inclination
[3/21] 正在处理 20130411_064800_TAI
[3] 已保存 Bxyz、Br、Btheta、Bphi、inclination
[4/21] 正在处理 20130411_070000_TAI
[4] 已保存 Bxyz、Br、Btheta、Bphi、inclination
[5/21] 正在处理 20130411_071200_TAI
[5] 已保存 Bxyz、Br、Btheta、Bphi、inclination
[6/21] 正在处理 20130411_072400_TAI
[6] 已保存 Bxyz、Br、Btheta、Bphi、inclination
[7/21] 正在处理 20130411_073600_TAI
[7] 已保存 Bxyz、Br、Btheta、Bphi、inclination
[8/21] 正在处理 20130411_074800_TAI
[8] 已保存 Bxyz、Br、Btheta、Bphi、inclination
[9/21] 正在处理 20130411_080000_TAI
[9] 已保存 Bxyz、Br、Btheta、Bphi、inclination
全部处理完成！文件名中的时间保留原始 TAI 标记。


In [2]:
import glob
ref_sub_map=sunpy.map.Map(r'E:\Learning\PHD3rd\sunspotscar\data\to_be_exam\20130411_065500_UTC\131sub_map\aia.lev1_euv_12s.2013-04-11T062510Z.131.image_lev1.fits')
# file_list=glob.glob(r'E:\Learning\PHD3rd\sunspotscar\data\to_be_exam\20130411_065500_UTC\inclination\*.fits')
# save_root=r'E:\Learning\PHD3rd\sunspotscar\data\to_be_exam\20130411_065500_UTC\inclination_submap'
file_list=glob.glob(r'E:\Learning\PHD3rd\sunspotscar\data\to_be_exam\20130411_065500_UTC\hmi_Br\*.fits')
save_root=r'E:\Learning\PHD3rd\sunspotscar\data\to_be_exam\20130411_065500_UTC\hmi_Br_submap'
for i in range(len(file_list)):
    file=file_list[i]
    map1=sunpy.map.Map(file)
    with propagate_with_solar_surface():
        rot_map1=map1.reproject_to(ref_sub_map.wcs,preserve_date_obs=True)
    for key in ['telescop', 'instrume', 'detector', 'wavelnth', 'waveunit',
                'bunit', 'exptime']:
        if key in map1.meta:
            rot_map1.meta[key] = map1.meta[key]
    rot_map1 = sunpy.map.Map(rot_map1.data.astype('float32'), rot_map1.meta)
    rot_map1.meta.pop('BLANK', None)
    rot_map1.save(os.path.join(save_root, os.path.basename(file_list[i]).replace('.fits','_submap.fits')),overwrite=True)

In [3]:
import glob
ref_sub_map=sunpy.map.Map(r'E:\Learning\PHD3rd\sunspotscar\data\to_be_exam\20130411_065500_UTC\hmi.Ic_45ssub_map\s.20130411_062445_TAI.2.continuum.fits')
# file_list=glob.glob(r'E:\Learning\PHD3rd\sunspotscar\data\to_be_exam\20130411_065500_UTC\inclination\*.fits')
# save_root=r'E:\Learning\PHD3rd\sunspotscar\data\to_be_exam\20130411_065500_UTC\inclination_submap'
file_list=glob.glob(r'E:\Learning\PHD3rd\sunspotscar\data\to_be_exam\20130411_065500_UTC\hmi.Ic_45ssub_map\*.fits')
save_root=r'E:\Learning\PHD3rd\sunspotscar\data\to_be_exam\20130411_065500_UTC\hmi.Ic_45ssub_map2'
for i in range(1,len(file_list)):
    file=file_list[i]
    map1=sunpy.map.Map(file)
    with propagate_with_solar_surface():
        rot_map1=map1.reproject_to(ref_sub_map.wcs,preserve_date_obs=True)
    for key in ['telescop', 'instrume', 'detector', 'wavelnth', 'waveunit',
                'bunit', 'exptime']:
        if key in map1.meta:
            rot_map1.meta[key] = map1.meta[key]
    rot_map1 = sunpy.map.Map(rot_map1.data.astype('float32'), rot_map1.meta)
    rot_map1.meta.pop('BLANK', None)
    rot_map1.save(os.path.join(save_root, os.path.basename(file_list[i]).replace('.fits','_submap.fits')),overwrite=True)

In [ ]:
#裁剪目标区域并进行旋转对齐
root_dir=r'D:\C'
file_time=os.listdir(root_dir)

#旋转裁剪对齐Br/inclination
for i in range(len(file_time)):
    try:
        ref_131=glob.glob(os.path.join(root_dir,file_time[i],'131sub_map','*.fits'))[0]
        ref_sub_map=sunpy.map.Map(ref_131)
        map1=sunpy.map.Map(os.path.join(root_dir,file_time[i],'hmi.B_720s','inclination_calc.fits'))
        with propagate_with_solar_surface():
            rot_map1=map1.reproject_to(ref_sub_map.wcs,preserve_date_obs=True)
        for key in ['telescop', 'instrume', 'detector', 'wavelnth', 'waveunit',
                    'bunit', 'exptime']:
            if key in map1.meta:
                rot_map1.meta[key] = map1.meta[key]
        rot_map1 = sunpy.map.Map(rot_map1.data.astype('float32'), rot_map1.meta)
        rot_map1.meta.pop('BLANK', None)
        rot_map1.save(os.path.join(root_dir,file_time[i],'hmi.B_720s','inclination_calc_sub.fits'), overwrite=True)
    except:
        print(file_time[i]+'hmi.B_720s'+'inclination_calc_sub.fits'+'保存失败')


In [4]:
root_dir=r'C:\Learning\PHD3rd\sunspotscar\scar_for_20240618'
aia_dir=os.path.join(root_dir,'AIA')
hmi_con_dir=os.path.join(root_dir,'HMI_continuum')

aia_94_list=glob.glob(os.path.join(aia_dir,'*94.image_lev1.fits'))
aia_131_list=glob.glob(os.path.join(aia_dir,'*131.image_lev1.fits'))
aia_171_list=glob.glob(os.path.join(aia_dir,'*171.image_lev1.fits'))
aia_211_list=glob.glob(os.path.join(aia_dir,'*211.image_lev1.fits'))
aia_304_list=glob.glob(os.path.join(aia_dir,'*304.image_lev1.fits'))
hmi_con_list=glob.glob(os.path.join(hmi_con_dir,'*continuum.fits'))
hmi_Br_list=glob.glob(os.path.join(root_dir,'hmi_Br','*Br.fits'))
inclination_list=glob.glob(os.path.join(root_dir,'inclination','*inclination.fits'))
fits_list=[aia_94_list, aia_131_list, aia_171_list, aia_211_list, aia_304_list, hmi_con_list, hmi_Br_list, inclination_list]

map0=sunpy.map.Map(r'C:\Learning\PHD3rd\sunspotscar\scar_for_20240618\AIA\aia.lev1_euv_12s.2024-06-18T211710Z.171.image_lev1.fits')
bottom_left=SkyCoord(Tx=100*u.arcsec,Ty=-540*u.arcsec,frame=map0.coordinate_frame)
top_right=SkyCoord(Tx=500*u.arcsec,Ty=-140*u.arcsec,frame=map0.coordinate_frame)
sub_map0=map0.submap(bottom_left,top_right=top_right)
sub_map0.meta.pop('blanks', None)

name_list=['94_submap','131_submap','171_submap','211_submap','304_submap','hmi_con_submap','hmi_Br_submap','inclination_submap']
for name in name_list:
    os.makedirs(os.path.join(root_dir,name),exist_ok=True)

for i in range(len(fits_list)):
    passbands=fits_list[i]
    save_root=os.path.join(root_dir,name_list[i])
    for fits_file in passbands:
        map1=sunpy.map.Map(fits_file)
        if map1==map0:
            rot_map1=sub_map0
        else:
            with propagate_with_solar_surface():
                rot_map1=map1.reproject_to(sub_map0.wcs,preserve_date_obs=True)
            for key in ['telescop', 'instrume', 'detector', 'wavelnth', 'waveunit',
                        'bunit', 'exptime']:
                if key in map1.meta:
                    rot_map1.meta[key] = map1.meta[key]
            rot_map1 = sunpy.map.Map(rot_map1.data.astype('float32'), rot_map1.meta)
            rot_map1.meta.pop('BLANK', None)
        rot_map1.save(os.path.join(save_root, os.path.basename(fits_file).replace('.fits','_submap.fits')),overwrite=True)
    
